# 🛠️ AI Software Engineering Assistant
### Capstone Project — OpenAI Agents SDK | Summer School '26

**Domain:** Software Development
**Problem statement:** An engineering assistant capable of analysing GitHub issues,
planning implementation, reviewing code, generating documentation, identifying bugs,
and assisting developers throughout the software development lifecycle (SDLC).

---

## What this notebook builds

A **multi-agent SDLC platform** with **6 specialised agents**, **7 tools**, agent
**handoffs**, shared **memory/context**, **structured outputs** (Pydantic), and a
**human-approval** gate on any action that touches the filesystem or executes code.

| # | Agent | Responsibility |
|---|-------|-----------------|
| 1 | Requirements Analysis Agent | Turns a raw issue/feature request into a structured spec + implementation plan |
| 2 | Coding Assistant Agent | Writes the actual code solution |
| 3 | Code Reviewer Agent | Reviews the code, flags risks, approves/rejects (self-review / reflection) |
| 4 | Testing Agent | Generates & runs unit tests, produces a structured test report |
| 5 | Documentation Writer Agent | Produces README / docstring style documentation |
| 6 | Bug Investigation Agent | Root-causes failing tests / bug reports, proposes a fix plan |

Plus a **Triage / Orchestrator Agent** that performs SDK-native **handoffs** to the
right specialist depending on the type of request (feature vs bug vs review vs docs).

### Advanced features implemented
- ✅ Planning & reasoning (Requirements Agent produces a structured plan)
- ✅ Reflection / self-review (Coder ⇄ Reviewer loop, up to N iterations)
- ✅ Parallel agent execution (`asyncio.gather` for Docs + Tests after review passes)
- ✅ Error handling & logging (Python `logging` → file + console)
- ✅ Session persistence (`SQLiteSession` — conversation survives notebook restarts)
- ✅ Long-term memory (JSON memory store per "project", separate from chat session)
- ✅ Human approval workflow (console confirmation before writing files / executing code)
- ✅ Real tool integration (live GitHub Issues REST API — no auth needed for search)

---

## How to run
1. Run cells top to bottom.
2. Add your `OPENAI_API_KEY` to **Colab Secrets** (🔑 icon in the left sidebar) before
   running the "Load API key" cell — see instructions in that cell.
3. The demo cells at the bottom will actually call the OpenAI API (small cost).


## 1. Install dependencies

In [ ]:
#@title Install packages
!pip install -q openai-agents pydantic nest_asyncio
print("✅ Dependencies installed")


## 2. Project folder structure

We create a small workspace on the Colab VM so agents have somewhere to read/write code, tests, docs and logs. Everything is created fresh under `/content`.

In [ ]:
#@title Create folders
import os

BASE_DIR = "/content/ai_swe_assistant"

FOLDERS = {
    "workspace": f"{BASE_DIR}/workspace",       # code the Coding Assistant writes
    "tests":     f"{BASE_DIR}/tests",           # generated unit tests
    "docs":      f"{BASE_DIR}/docs",            # generated documentation
    "outputs":   f"{BASE_DIR}/outputs",         # final structured reports (JSON)
    "logs":      f"{BASE_DIR}/logs",            # run logs
    "memory":    f"{BASE_DIR}/memory",          # long-term project memory + session db
    "diagrams":  f"{BASE_DIR}/diagrams",        # architecture diagram
}

for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"📁 {name:10s} -> {path}")


## 3. Load your OpenAI API key

**Recommended (secure) way in Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar.
2. Add a new secret named `OPENAI_API_KEY` with your key as the value.
3. Toggle "Notebook access" on for this secret.
4. Run the cell below — it will read the secret automatically.

If you're not running in Colab (or don't have a secret set), it falls back to an
environment variable and finally to a manual `input()` prompt.


In [ ]:
#@title Load API key
import os

api_key = None

# 1) Try Colab userdata secret
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
    print("✅ Loaded OPENAI_API_KEY from Colab secrets")
except Exception:
    pass

# 2) Fall back to environment variable
if not api_key:
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("✅ Loaded OPENAI_API_KEY from environment variable")

# 3) Fall back to manual input (last resort — not recommended, key is visible)
if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your OPENAI_API_KEY: ")

os.environ["OPENAI_API_KEY"] = api_key
assert api_key, "An OpenAI API key is required to run this notebook."
print("🔐 API key is set for this session.")


## 4. Imports & logging setup

All error handling / logging in this notebook flows through a single logger that writes to both the console and a rotating log file in `logs/`.

In [ ]:
#@title Core imports
import asyncio
import json
import logging
import subprocess
import sys
import textwrap
import traceback
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal, Optional

import nest_asyncio
import requests
from pydantic import BaseModel, Field

from agents import (
    Agent,
    Runner,
    RunContextWrapper,
    RunResult,
    function_tool,
    handoff,
)
from agents.memory.sqlite_session import SQLiteSession

# Colab / Jupyter already runs an event loop -> patch so we can use asyncio.run()
nest_asyncio.apply()

print("✅ Imports OK")


In [ ]:
#@title Logging configuration
LOG_PATH = f"{FOLDERS['logs']}/run.log"

logger = logging.getLogger("ai_swe_assistant")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")

_file_handler = logging.FileHandler(LOG_PATH)
_file_handler.setFormatter(_fmt)
logger.addHandler(_file_handler)

_stream_handler = logging.StreamHandler(sys.stdout)
_stream_handler.setFormatter(_fmt)
logger.addHandler(_stream_handler)

logger.info("Logger initialised. Writing to %s", LOG_PATH)


## 5. Structured outputs (Pydantic schemas)

Each agent returns a typed Pydantic object (via `output_type=`) instead of free text,
so downstream agents / code can consume the result programmatically instead of
re-parsing prose.

In [ ]:
#@title Pydantic output schemas
class ImplementationStep(BaseModel):
    step_number: int
    description: str
    owner_agent: str = Field(description="Which specialist agent should perform this step")

class RequirementsSpec(BaseModel):
    """Structured output of the Requirements Analysis Agent."""
    title: str
    problem_summary: str
    functional_requirements: list[str]
    non_functional_requirements: list[str] = Field(default_factory=list)
    acceptance_criteria: list[str]
    implementation_plan: list[ImplementationStep]
    related_github_issues: list[str] = Field(default_factory=list)
    risk_notes: Optional[str] = None


class CodeSolution(BaseModel):
    """Structured output of the Coding Assistant Agent."""
    filename: str
    language: str
    code: str
    explanation: str
    lint_status: str


class CodeReviewResult(BaseModel):
    """Structured output of the Code Reviewer Agent."""
    approved: bool
    severity: Literal["none", "low", "medium", "high", "critical"]
    issues_found: list[str] = Field(default_factory=list)
    suggestions: list[str] = Field(default_factory=list)
    summary: str


class TestCaseResult(BaseModel):
    name: str
    passed: bool
    details: str = ""

class TestReport(BaseModel):
    """Structured output of the Testing Agent."""
    test_filename: str
    test_code: str
    total_tests: int
    passed: int
    failed: int
    results: list[TestCaseResult]
    all_passed: bool


class DocumentationOutput(BaseModel):
    """Structured output of the Documentation Writer Agent."""
    doc_filename: str
    markdown_content: str
    summary: str


class BugInvestigationReport(BaseModel):
    """Structured output of the Bug Investigation Agent."""
    root_cause: str
    affected_files: list[str]
    proposed_fix: str
    confidence: Literal["low", "medium", "high"]
    related_github_issues: list[str] = Field(default_factory=list)


print("✅ Structured output schemas defined:")
for m in [RequirementsSpec, CodeSolution, CodeReviewResult, TestReport,
          DocumentationOutput, BugInvestigationReport]:
    print(" -", m.__name__)


## 6. Shared context & long-term memory

Two layers of "memory" are used, matching the assignment's *Memory/context management*
requirement:

1. **`EngineeringContext`** — a Python dataclass passed to every `Runner.run(..., context=...)`
   call. Agents/tools can read *and write* to it during a run (short-term / working memory).
2. **`ProjectMemoryStore`** — a small JSON-file-backed long-term memory that persists
   across runs (and across notebook restarts), keyed by project name.
3. **`SQLiteSession`** — the SDK's own conversation-history session, so the *dialogue*
   with the Triage Agent persists turn-to-turn.


In [ ]:
#@title Context, long-term memory & session
@dataclass
class EngineeringContext:
    project_name: str
    repo_full_name: str = "octocat/Hello-World"   # owner/repo used for GitHub tool calls
    workspace_dir: str = FOLDERS["workspace"]
    run_log: list[dict] = field(default_factory=list)   # in-run working memory
    approved_actions: set = field(default_factory=set)

    def remember(self, event: str, **details):
        entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": event,
            **details,
        }
        self.run_log.append(entry)
        logger.info("MEMORY | %s | %s", event, details)


class ProjectMemoryStore:
    """Very small long-term memory: one JSON file per project under memory/."""

    def __init__(self, memory_dir: str):
        self.memory_dir = Path(memory_dir)
        self.memory_dir.mkdir(parents=True, exist_ok=True)

    def _path(self, project_name: str) -> Path:
        safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in project_name)
        return self.memory_dir / f"{safe}.json"

    def load(self, project_name: str) -> dict:
        p = self._path(project_name)
        if p.exists():
            return json.loads(p.read_text())
        return {"project_name": project_name, "history": []}

    def append(self, project_name: str, record: dict):
        data = self.load(project_name)
        record = {"timestamp": datetime.now(timezone.utc).isoformat(), **record}
        data["history"].append(record)
        self._path(project_name).write_text(json.dumps(data, indent=2, default=str))
        logger.info("Long-term memory updated for project '%s'", project_name)


memory_store = ProjectMemoryStore(FOLDERS["memory"])

# SDK-native conversation session (persists chat turns for the Triage Agent)
chat_session = SQLiteSession(
    session_id="ai_swe_assistant_main",
    db_path=f"{FOLDERS['memory']}/session.db",
)

print("✅ Context, long-term memory store, and SQLiteSession ready")


## 7. Human-in-the-loop approval

Any tool that **writes to disk** or **executes code** must be explicitly approved by
a human before it runs. In Colab, this pauses the cell and asks for `y/n` via `input()`.
Set `AUTO_APPROVE = True` below to skip prompts for a fully unattended demo run
(useful when re-running the notebook for grading).

In [ ]:
#@title Approval gate
AUTO_APPROVE = False  #@param {type:"boolean"}

def request_human_approval(action: str, details: str) -> bool:
    """Block and ask a human to approve a sensitive action.

    Returns True if approved, False otherwise. Every decision is logged.
    """
    print("\n" + "=" * 70)
    print(f"🧑‍💻 HUMAN APPROVAL REQUIRED: {action}")
    print("-" * 70)
    print(textwrap.shorten(details, width=500, placeholder=" ...[truncated]"))
    print("=" * 70)

    if AUTO_APPROVE:
        logger.warning("AUTO_APPROVE is ON — action '%s' approved automatically.", action)
        return True

    answer = input("Approve this action? [y/N]: ").strip().lower()
    approved = answer == "y"
    logger.info("Human decision for '%s': %s", action, "APPROVED" if approved else "REJECTED")
    return approved


## 8. Tools

**7 tools** available to the agents, covering file I/O, code execution/linting, a
live external API (GitHub Issues search), and documentation persistence. Tools that
mutate state go through the human-approval gate above.

In [ ]:
#@title Tool 1 & 2 — read/write files in the workspace
@function_tool
def read_code_file(path: str) -> str:
    """Read a file from the project workspace.

    Args:
        path: filename (relative to the workspace folder) to read.
    """
    full_path = Path(FOLDERS["workspace"]) / path
    if not full_path.exists():
        return f"ERROR: file '{path}' does not exist in workspace."
    try:
        content = full_path.read_text()
        logger.info("Tool read_code_file: read %s (%d chars)", path, len(content))
        return content
    except Exception as e:
        logger.error("read_code_file failed: %s", e)
        return f"ERROR reading file: {e}"


@function_tool
def write_code_file(path: str, content: str) -> str:
    """Write (or overwrite) a file in the project workspace. Requires human approval.

    Args:
        path: filename (relative to workspace folder) to write.
        content: full text content to write to the file.
    """
    approved = request_human_approval(
        action=f"Write file '{path}'",
        details=f"About to write {len(content)} characters to workspace/{path}\n\nPreview:\n{content[:300]}",
    )
    if not approved:
        logger.warning("write_code_file REJECTED for %s", path)
        return "REJECTED: human did not approve this file write."

    full_path = Path(FOLDERS["workspace"]) / path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(content)
    logger.info("Tool write_code_file: wrote %s (%d chars)", path, len(content))
    return f"OK: wrote {len(content)} characters to workspace/{path}"


In [ ]:
#@title Tool 3 — lint Python code (static, no approval needed — read-only)
@function_tool
def lint_python_code(code: str) -> str:
    """Statically check Python source code for syntax errors using compile().

    Args:
        code: the Python source code to check.
    """
    try:
        compile(code, "<agent_code>", "exec")
        logger.info("Tool lint_python_code: OK, no syntax errors")
        return "OK: no syntax errors detected."
    except SyntaxError as e:
        msg = f"SyntaxError: {e.msg} at line {e.lineno}, offset {e.offset}"
        logger.warning("Tool lint_python_code: %s", msg)
        return msg
    except Exception as e:
        logger.error("lint_python_code failed: %s", e)
        return f"ERROR during lint: {e}"


In [ ]:
#@title Tool 4 — execute Python tests in a sandboxed subprocess (requires approval)
@function_tool
def run_python_tests(test_filename: str, test_code: str, target_filename: str) -> str:
    """Save and execute a pytest-style test file against code already in the workspace,
    in an isolated subprocess with a timeout. Requires human approval before execution.

    Args:
        test_filename: filename to save the test file as (inside tests/ folder).
        test_code: full pytest source code (should import the target module).
        target_filename: the source file under test (must already exist in workspace/).
    """
    approved = request_human_approval(
        action=f"Execute test suite '{test_filename}'",
        details=f"About to run pytest on {test_filename} against workspace/{target_filename} "
                f"in a subprocess with a 30s timeout.\n\nTest code preview:\n{test_code[:300]}",
    )
    if not approved:
        return "REJECTED: human did not approve test execution."

    tests_dir = Path(FOLDERS["tests"])
    tests_dir.mkdir(parents=True, exist_ok=True)
    test_path = tests_dir / test_filename
    test_path.write_text(test_code)

    workspace_dir = Path(FOLDERS["workspace"])
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pytest", str(test_path), "-v", "--tb=short"],
            cwd=str(workspace_dir),
            capture_output=True,
            text=True,
            timeout=30,
            env={**__import__("os").environ, "PYTHONPATH": str(workspace_dir)},
        )
        output = (result.stdout + "\n" + result.stderr)[-4000:]
        logger.info("Tool run_python_tests: exit code %s", result.returncode)
        return f"EXIT_CODE={result.returncode}\n{output}"
    except subprocess.TimeoutExpired:
        logger.error("run_python_tests timed out for %s", test_filename)
        return "ERROR: test execution timed out after 30s."
    except Exception as e:
        logger.error("run_python_tests failed: %s", e)
        return f"ERROR running tests: {e}"


In [ ]:
#@title Tool 5 — search live GitHub issues (real external API, no auth needed for low-volume search)
@function_tool
def search_github_issues(repo_full_name: str, query: str) -> str:
    """Search real GitHub issues for a repository using the public GitHub REST API.

    Args:
        repo_full_name: repository as 'owner/repo', e.g. 'psf/requests'.
        query: free-text search terms (combined with the repo filter).
    """
    try:
        resp = requests.get(
            "https://api.github.com/search/issues",
            params={"q": f"repo:{repo_full_name} {query}", "per_page": 5},
            headers={"Accept": "application/vnd.github+json"},
            timeout=15,
        )
        if resp.status_code != 200:
            logger.warning("GitHub search non-200: %s %s", resp.status_code, resp.text[:200])
            return f"GitHub API returned status {resp.status_code}: {resp.text[:200]}"

        data = resp.json()
        items = data.get("items", [])[:5]
        if not items:
            return f"No issues found for query '{query}' in {repo_full_name}."

        lines = [f"Found {data.get('total_count', len(items))} matching issue(s) (showing top {len(items)}):"]
        for it in items:
            lines.append(f"- #{it['number']} [{it['state']}] {it['title']} -> {it['html_url']}")
        logger.info("Tool search_github_issues: %d results for '%s'", len(items), query)
        return "\n".join(lines)
    except Exception as e:
        logger.error("search_github_issues failed: %s", e)
        return f"ERROR calling GitHub API: {e}"


In [ ]:
#@title Tool 6 — save documentation to disk (requires approval)
@function_tool
def save_documentation(filename: str, markdown_content: str) -> str:
    """Save generated documentation (Markdown) to the docs/ folder. Requires human approval.

    Args:
        filename: filename to save as (e.g. 'README.md').
        markdown_content: the full Markdown content to save.
    """
    approved = request_human_approval(
        action=f"Save documentation '{filename}'",
        details=f"About to write {len(markdown_content)} characters to docs/{filename}\n\n"
                f"Preview:\n{markdown_content[:300]}",
    )
    if not approved:
        return "REJECTED: human did not approve saving documentation."

    doc_path = Path(FOLDERS["docs"]) / filename
    doc_path.parent.mkdir(parents=True, exist_ok=True)
    doc_path.write_text(markdown_content)
    logger.info("Tool save_documentation: wrote %s", filename)
    return f"OK: saved documentation to docs/{filename}"


In [ ]:
#@title Tool 7 — persist a structured report to outputs/ (read/write, low risk, no approval)
@function_tool
def save_structured_report(report_name: str, report_json: str) -> str:
    """Persist a structured JSON report (e.g. a review or test report) to outputs/.

    Args:
        report_name: filename to save as (e.g. 'code_review.json').
        report_json: the JSON string content to save.
    """
    out_path = Path(FOLDERS["outputs"]) / report_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(report_json)
    logger.info("Tool save_structured_report: wrote %s", report_name)
    return f"OK: saved report to outputs/{report_name}"


ALL_TOOLS_SUMMARY = [
    "read_code_file", "write_code_file", "lint_python_code",
    "run_python_tests", "search_github_issues", "save_documentation",
    "save_structured_report",
]
print(f"✅ {len(ALL_TOOLS_SUMMARY)} tools defined:", ALL_TOOLS_SUMMARY)


## 9. The 6 specialised agents

Each agent gets a narrow role, only the tools it needs, and a Pydantic `output_type`
so its result is machine-readable by the pipeline / the next agent.

In [ ]:
#@title Agent 1 — Requirements Analysis Agent
requirements_agent = Agent[EngineeringContext](
    name="Requirements Analysis Agent",
    handoff_description="Analyses a raw issue / feature request and produces a structured spec and plan.",
    instructions=(
        "You are a senior software requirements analyst. Given a raw GitHub issue or "
        "feature request, produce a structured requirements specification: a clear "
        "problem summary, functional and non-functional requirements, measurable "
        "acceptance criteria, and a step-by-step implementation plan assigning each "
        "step to the right downstream specialist agent "
        "(one of: 'Coding Assistant Agent', 'Code Reviewer Agent', 'Testing Agent', "
        "'Documentation Writer Agent', 'Bug Investigation Agent'). "
        "Use the search_github_issues tool to check whether similar issues already "
        "exist in the repository before finalising the plan."
    ),
    tools=[search_github_issues],
    output_type=RequirementsSpec,
)


In [ ]:
#@title Agent 2 — Coding Assistant Agent
coding_agent = Agent[EngineeringContext](
    name="Coding Assistant Agent",
    handoff_description="Writes Python code to implement a given requirement or fix.",
    instructions=(
        "You are an expert Python software engineer. Given a requirement or bug fix "
        "description, write clean, well-documented, idiomatic Python code that "
        "implements it. Always call lint_python_code on your code before returning, "
        "and fix any syntax errors it reports. Only call write_code_file if you need "
        "to persist the file to the shared workspace (this requires human approval, "
        "so only do it when explicitly asked to save). Return a CodeSolution with the "
        "final code, filename, language, an explanation of your approach, and the "
        "lint status you observed."
    ),
    tools=[lint_python_code, write_code_file, read_code_file],
    output_type=CodeSolution,
)


In [ ]:
#@title Agent 3 — Code Reviewer Agent  (also used for the reflection / self-review loop)
code_reviewer_agent = Agent[EngineeringContext](
    name="Code Reviewer Agent",
    handoff_description="Reviews code for correctness, style, security and maintainability issues.",
    instructions=(
        "You are a meticulous senior code reviewer. Review the given Python code for "
        "correctness, edge cases, security issues, readability, and adherence to "
        "PEP 8. Use lint_python_code to double check for syntax errors. Be strict: "
        "only set approved=True if there are no medium/high/critical issues. Return a "
        "CodeReviewResult with your verdict, a severity rating, a list of concrete "
        "issues found, actionable suggestions, and a short summary."
    ),
    tools=[lint_python_code, read_code_file],
    output_type=CodeReviewResult,
)


In [ ]:
#@title Agent 4 — Testing Agent
testing_agent = Agent[EngineeringContext](
    name="Testing Agent",
    handoff_description="Writes and executes unit tests for a given piece of code.",
    instructions=(
        "You are a test engineer. Given a Python code solution, write a thorough "
        "pytest test suite covering normal cases, edge cases, and error handling. "
        "Then call run_python_tests to actually execute the suite against the code "
        "(this requires human approval — call it once you're confident in the tests). "
        "Parse the subprocess output to determine how many tests passed/failed, and "
        "return a TestReport with the test code, per-test results, and whether "
        "all tests passed."
    ),
    tools=[run_python_tests, read_code_file],
    output_type=TestReport,
)


In [ ]:
#@title Agent 5 — Documentation Writer Agent
documentation_agent = Agent[EngineeringContext](
    name="Documentation Writer Agent",
    handoff_description="Writes Markdown documentation (README-style) for a piece of code.",
    instructions=(
        "You are a technical writer. Given a code solution and its requirements, "
        "write clear Markdown documentation: what the code does, how to install/run "
        "it, a usage example, and any important caveats. Call save_documentation to "
        "persist it (requires human approval). Return a DocumentationOutput with the "
        "filename, full markdown content, and a one-line summary."
    ),
    tools=[save_documentation, read_code_file],
    output_type=DocumentationOutput,
)


In [ ]:
#@title Agent 6 — Bug Investigation Agent
bug_investigation_agent = Agent[EngineeringContext](
    name="Bug Investigation Agent",
    handoff_description="Investigates failing tests or bug reports and proposes a root-cause fix.",
    instructions=(
        "You are a debugging specialist. Given a bug report or a failing test report, "
        "investigate the likely root cause by reading the relevant source with "
        "read_code_file and, if useful, searching search_github_issues for similar, "
        "previously-reported issues in the repo. Propose a concrete fix. Return a "
        "BugInvestigationReport with the root cause, affected files, proposed fix, "
        "your confidence level, and any related GitHub issues you found."
    ),
    tools=[read_code_file, search_github_issues, lint_python_code],
    output_type=BugInvestigationReport,
)

print("✅ All 6 specialist agents created")


## 10. Triage / Orchestrator Agent — SDK-native handoffs

This agent doesn't solve anything itself — it reads the user's request and **hands off**
to the correct specialist using the SDK's built-in `handoffs=[...]` mechanism. This
demonstrates the *"Agent interaction and handoff flow"* requirement directly (as opposed
to the manually-orchestrated pipeline in the next section, which composes multiple
specialists together for a full SDLC run).

In [ ]:
#@title Triage Agent with handoffs
def _log_handoff(agent_name):
    def _on_handoff(ctx: RunContextWrapper[EngineeringContext]):
        ctx.context.remember("handoff", to_agent=agent_name)
        logger.info("🔀 Handoff -> %s", agent_name)
    return _on_handoff


triage_agent = Agent[EngineeringContext](
    name="Triage Agent",
    instructions=(
        "You are the entry point of an AI Software Engineering Assistant. Read the "
        "user's request and hand off to exactly one specialist:\n"
        "- New feature / issue to analyse -> Requirements Analysis Agent\n"
        "- 'write/implement code for ...' -> Coding Assistant Agent\n"
        "- 'review this code ...' -> Code Reviewer Agent\n"
        "- 'write/run tests for ...' -> Testing Agent\n"
        "- 'write docs/README for ...' -> Documentation Writer Agent\n"
        "- 'investigate this bug / failing test ...' -> Bug Investigation Agent\n"
        "Always hand off — do not try to answer the request yourself."
    ),
    handoffs=[
        handoff(requirements_agent, on_handoff=_log_handoff("Requirements Analysis Agent")),
        handoff(coding_agent, on_handoff=_log_handoff("Coding Assistant Agent")),
        handoff(code_reviewer_agent, on_handoff=_log_handoff("Code Reviewer Agent")),
        handoff(testing_agent, on_handoff=_log_handoff("Testing Agent")),
        handoff(documentation_agent, on_handoff=_log_handoff("Documentation Writer Agent")),
        handoff(bug_investigation_agent, on_handoff=_log_handoff("Bug Investigation Agent")),
    ],
)

print("✅ Triage Agent ready with 6 handoff targets")


## 11. Demo A — Triage handoff flow (interactive / single request)

Ask the Triage Agent one natural-language request. It will hand off to the right
specialist and you'll get a structured, typed result back. Conversation turns are
persisted in `chat_session` (`SQLiteSession`), so context carries over if you run this
cell again with a follow-up request.

In [ ]:
#@title Run a single request through the Triage Agent
async def ask_assistant(user_request: str, project_name: str = "demo-project") -> RunResult:
    ctx = EngineeringContext(project_name=project_name)
    logger.info("=== Triage run: %s ===", user_request)
    try:
        result = await Runner.run(
            triage_agent,
            input=user_request,
            context=ctx,
            session=chat_session,
            max_turns=15,
        )
        memory_store.append(project_name, {
            "request": user_request,
            "final_agent": result.last_agent.name,
            "output": str(result.final_output),
        })
        return result
    except Exception as e:
        logger.error("Triage run failed: %s\n%s", e, traceback.format_exc())
        raise


# Example — change this and re-run to try other request types
example_request = (
    "We have a repo 'psf/requests'. Please analyse this feature request: "
    "'Add a helper function that validates whether a given string is a syntactically "
    "correct Python identifier, and returns a clear error message if not.'"
)

result = asyncio.run(ask_assistant(example_request))

print("\n🏁 Handled by:", result.last_agent.name)
print("\n📦 Structured output:\n")
print(result.final_output.model_dump_json(indent=2) if hasattr(result.final_output, "model_dump_json")
      else result.final_output)


## 12. Reflection loop — Coder ⇄ Reviewer

Instead of trusting the Coding Assistant's first draft, we loop it through the
Code Reviewer Agent up to `MAX_REFLECTION_ROUNDS` times, feeding the reviewer's
feedback back into the coder, until the review is `approved=True` or we run out of
rounds. This is the *reflection / self-review* advanced feature.

In [ ]:
#@title Reflection loop implementation
MAX_REFLECTION_ROUNDS = 2

async def code_with_reflection(spec: RequirementsSpec, ctx: EngineeringContext):
    """Generate code, review it, and iterate until approved or rounds are exhausted."""
    prompt = (
        f"Implement this requirement:\n\nTitle: {spec.title}\n"
        f"Summary: {spec.problem_summary}\n"
        f"Functional requirements:\n- " + "\n- ".join(spec.functional_requirements)
    )

    code_result: Optional[CodeSolution] = None
    review_result: Optional[CodeReviewResult] = None

    for round_num in range(1, MAX_REFLECTION_ROUNDS + 1):
        logger.info("🔁 Reflection round %d/%d", round_num, MAX_REFLECTION_ROUNDS)

        coder_run = await Runner.run(coding_agent, input=prompt, context=ctx, max_turns=10)
        code_result = coder_run.final_output
        ctx.remember("code_generated", round=round_num, filename=code_result.filename)

        review_prompt = (
            f"Review this code (filename: {code_result.filename}):\n\n"
            f"```{code_result.language}\n{code_result.code}\n```\n\n"
            f"Context / explanation from the author: {code_result.explanation}"
        )
        reviewer_run = await Runner.run(code_reviewer_agent, input=review_prompt, context=ctx, max_turns=10)
        review_result = reviewer_run.final_output
        ctx.remember("code_reviewed", round=round_num, approved=review_result.approved,
                     severity=review_result.severity)

        if review_result.approved:
            logger.info("✅ Code approved on round %d", round_num)
            break
        else:
            logger.warning("❌ Round %d rejected (severity=%s): %s",
                            round_num, review_result.severity, review_result.summary)
            # feed reviewer feedback back into the next coding attempt
            prompt = (
                f"{prompt}\n\nYour previous attempt was reviewed and REJECTED "
                f"(severity={review_result.severity}). Issues found:\n- "
                + "\n- ".join(review_result.issues_found)
                + "\n\nSuggestions:\n- " + "\n- ".join(review_result.suggestions)
                + "\n\nPlease produce a corrected version that addresses all of these."
            )

    return code_result, review_result

print("✅ Reflection loop ready")


## 13. Full SDLC pipeline (sequential + parallel + error handling)

This function chains all six agents into one coherent workflow, mirroring a real
engineering lifecycle:

```
Requirements  ->  Coding ⇄ Review (reflection loop)  ->  ┌ Testing        ┐  (parallel)
                                                          └ Documentation  ┘
                                                                 |
                                            (if tests fail) -> Bug Investigation
```

Every stage is wrapped in `try/except` with logging, and every result is written to
both the run-scoped context (`ctx.remember`) and the long-term `memory_store`.

In [ ]:
#@title Full pipeline
async def run_full_sdlc_pipeline(issue_text: str, repo_full_name: str, project_name: str):
    ctx = EngineeringContext(project_name=project_name, repo_full_name=repo_full_name)
    pipeline_report = {"project_name": project_name, "stages": {}}

    try:
        # ---- Stage 1: Requirements ------------------------------------------------
        logger.info("STAGE 1/5: Requirements Analysis")
        req_prompt = f"Repository: {repo_full_name}\n\nIssue / request:\n{issue_text}"
        req_run = await Runner.run(requirements_agent, input=req_prompt, context=ctx, max_turns=10)
        spec: RequirementsSpec = req_run.final_output
        ctx.remember("requirements_ready", title=spec.title)
        pipeline_report["stages"]["requirements"] = spec.model_dump()

        # ---- Stage 2: Coding + Reflection review -----------------------------------
        logger.info("STAGE 2/5: Coding + Review (reflection loop)")
        code_result, review_result = await code_with_reflection(spec, ctx)
        pipeline_report["stages"]["code"] = code_result.model_dump()
        pipeline_report["stages"]["review"] = review_result.model_dump()

        # ---- Stage 3 & 4: Testing + Documentation IN PARALLEL ----------------------
        logger.info("STAGE 3&4/5: Testing + Documentation (parallel)")
        test_prompt = (
            f"Write and run pytest tests for this code "
            f"(filename: {code_result.filename}):\n\n```{code_result.language}\n{code_result.code}\n```"
        )
        doc_prompt = (
            f"Write documentation for this code, given the original requirements.\n\n"
            f"Requirements title: {spec.title}\nSummary: {spec.problem_summary}\n\n"
            f"Code (filename: {code_result.filename}):\n```{code_result.language}\n{code_result.code}\n```"
        )

        test_task = Runner.run(testing_agent, input=test_prompt, context=ctx, max_turns=10)
        doc_task = Runner.run(documentation_agent, input=doc_prompt, context=ctx, max_turns=10)
        test_run, doc_run = await asyncio.gather(test_task, doc_task, return_exceptions=True)

        if isinstance(test_run, Exception):
            logger.error("Testing stage failed: %s", test_run)
            pipeline_report["stages"]["tests"] = {"error": str(test_run)}
            test_report = None
        else:
            test_report: TestReport = test_run.final_output
            pipeline_report["stages"]["tests"] = test_report.model_dump()
            ctx.remember("tests_run", all_passed=test_report.all_passed)

        if isinstance(doc_run, Exception):
            logger.error("Documentation stage failed: %s", doc_run)
            pipeline_report["stages"]["documentation"] = {"error": str(doc_run)}
        else:
            doc_result: DocumentationOutput = doc_run.final_output
            pipeline_report["stages"]["documentation"] = doc_result.model_dump()

        # ---- Stage 5: Bug investigation, only if tests failed ---------------------
        if test_report is not None and not test_report.all_passed:
            logger.info("STAGE 5/5: Bug Investigation (tests failed)")
            bug_prompt = (
                f"Tests failed for {code_result.filename}. Test report:\n"
                f"{json.dumps(test_report.model_dump(), indent=2)}\n\n"
                f"Source code:\n```{code_result.language}\n{code_result.code}\n```"
            )
            bug_run = await Runner.run(bug_investigation_agent, input=bug_prompt, context=ctx, max_turns=10)
            bug_report: BugInvestigationReport = bug_run.final_output
            pipeline_report["stages"]["bug_investigation"] = bug_report.model_dump()
            ctx.remember("bug_investigated", root_cause=bug_report.root_cause)
        else:
            logger.info("STAGE 5/5: skipped (no failing tests, or tests stage unavailable)")

        pipeline_report["run_log"] = ctx.run_log
        pipeline_report["status"] = "completed"

    except Exception as e:
        logger.error("Pipeline failed: %s\n%s", e, traceback.format_exc())
        pipeline_report["status"] = "failed"
        pipeline_report["error"] = str(e)

    # Persist to long-term memory + outputs/
    memory_store.append(project_name, {"pipeline_status": pipeline_report["status"]})
    out_path = Path(FOLDERS["outputs"]) / f"{project_name}_pipeline_report.json"
    out_path.write_text(json.dumps(pipeline_report, indent=2, default=str))
    logger.info("Pipeline report saved to %s", out_path)

    return pipeline_report

print("✅ Full SDLC pipeline function ready")


## 14. Demo B — run the full pipeline end-to-end

This is the main demo: a realistic feature request runs through **all 6 agents**,
with real GitHub lookups, human approval prompts for file writes and test execution,
a reflection loop between the coder and reviewer, and parallel testing + docs.

> 💡 Tip: set `AUTO_APPROVE = True` in the approval-gate cell above if you want this
> to run unattended (e.g. for a screen-recorded demo video).

In [ ]:
#@title Run the full SDLC pipeline demo
demo_issue = (
    "Users have requested a small utility function `is_valid_identifier(s: str) -> bool` "
    "that checks whether a string is a syntactically valid Python identifier "
    "(letters/digits/underscores, not starting with a digit, not a reserved keyword), "
    "with clear docstrings and full test coverage."
)

pipeline_report = asyncio.run(
    run_full_sdlc_pipeline(
        issue_text=demo_issue,
        repo_full_name="psf/requests",
        project_name="identifier-validator",
    )
)

print("\n\n================ PIPELINE SUMMARY ================")
print("Status:", pipeline_report["status"])
for stage_name in pipeline_report["stages"]:
    print(f" - stage completed: {stage_name}")


## 15. Demo C — Bug Investigation Agent in isolation

A quick standalone demo showing the Bug Investigation Agent reasoning about a bug report and cross-checking GitHub for related issues, independent of the full pipeline.

In [ ]:
#@title Standalone bug investigation demo
bug_report_text = (
    "Bug report: calling requests.get() with a malformed URL sometimes raises an "
    "unhelpful low-level exception instead of a clear requests.exceptions error. "
    "Users are confused about what went wrong."
)

async def run_bug_demo():
    ctx = EngineeringContext(project_name="bug-demo", repo_full_name="psf/requests")
    run = await Runner.run(bug_investigation_agent, input=bug_report_text, context=ctx, max_turns=10)
    return run.final_output

bug_result = asyncio.run(run_bug_demo())
print(bug_result.model_dump_json(indent=2))


## 16. Architecture diagram

A Mermaid diagram describing the agent architecture, saved to `diagrams/` as both
`.mmd` (source) and embedded here for viewing. Paste the `.mmd` content into
[mermaid.live](https://mermaid.live) or a Markdown renderer that supports Mermaid to
export a PNG/SVG for your presentation slides.

In [ ]:
#@title Generate & save architecture diagram
architecture_mermaid = r"""flowchart TD
    U[User / GitHub Issue] --> T[Triage Agent]

    T -- feature/issue --> RA[Requirements Analysis Agent]
    T -- write code --> CA[Coding Assistant Agent]
    T -- review code --> CR[Code Reviewer Agent]
    T -- write/run tests --> TA[Testing Agent]
    T -- write docs --> DA[Documentation Writer Agent]
    T -- investigate bug --> BA[Bug Investigation Agent]

    subgraph Full SDLC Pipeline (orchestrated)
        RA --> CA
        CA <-->|reflection loop| CR
        CR --> TA
        CR --> DA
        TA -- tests fail --> BA
    end

    RA -. tool .-> GH[(GitHub Issues API)]
    BA -. tool .-> GH
    CA -. tool .-> LINT[Lint tool]
    CA -. tool .-> WF[[write_code_file *approval*]]
    TA -. tool .-> RT[[run_python_tests *approval*]]
    DA -. tool .-> SD[[save_documentation *approval*]]

    CTX[(EngineeringContext\nshort-term memory)] -.-> RA & CA & CR & TA & DA & BA
    MEM[(ProjectMemoryStore\nlong-term memory, JSON)] -.-> T
    SESS[(SQLiteSession\nconversation persistence)] -.-> T
"""

diagram_path = Path(FOLDERS["diagrams"]) / "architecture.mmd"
diagram_path.write_text(architecture_mermaid)
print(f"✅ Saved architecture diagram to {diagram_path}")
print("\nPreview:\n")
print(architecture_mermaid)


## 17. Inspect everything this run produced

In [ ]:
#@title List all generated artifacts
def print_tree(base: str):
    base_path = Path(base)
    for p in sorted(base_path.rglob("*")):
        if p.is_file():
            rel = p.relative_to(base_path)
            size = p.stat().st_size
            print(f"  {rel}  ({size} bytes)")

print("📂 Generated project artifacts:\n")
print_tree(BASE_DIR)

print("\n📝 Last 20 log lines:\n")
log_lines = Path(LOG_PATH).read_text().splitlines()
for line in log_lines[-20:]:
    print(line)
